# Mag Vector — quaternionischer Differentialoperator

Notebook für das **Bamberger Modell** (**SageMath 10.9**). Kernel in Jupyter/Cursor: **`sagemath-10.9`** (Anzeige „SageMath 10.9“; nicht nur „Python 3“). Sage 10.9 (Upstream): [Release Tour](https://github.com/sagemath/sage/wiki/Sage-10.9-Release-Tour).

- In Sage können Sie Potenzen als `**` oder mit Caret schreiben (Preparser).
- Alternativ im Sage-Terminal: `load("Mag Vector.py")` bzw. `attach("Mag Vector.py")` nach Änderungen.

## Imports und Quaternionen-Raum

In [ ]:
from sage.all import SR, QuaternionAlgebra, diff, var

# 1. Symbolische Variablen für Raum und Zeit
t, x, y, z, c = var("t x y z c")

# 2. Hamilton-Quaternionen über dem symbolischen Ring
Q = QuaternionAlgebra(SR, -1, -1)
i, j, k = Q.gens()

## Hilfsfunktion `_simplify_quaternion_element`

In [ ]:
def _simplify_quaternion_element(F):
    """Vereinfacht F best effort (API je nach Sage-Version unterschiedlich)."""
    try:
        return F.canonical_form()
    except AttributeError:
        pass
    try:
        return F.reduce()
    except AttributeError:
        pass
    try:
        basis = F.parent().basis()
        return sum(comp.simplify_full() * b for comp, b in zip(F.coefficient_tuple(), basis))
    except Exception:
        return F

## Operator `apply_quaternion_operator`

In [ ]:
def apply_quaternion_operator(Phi, Ax, Ay, Az):
    """
    Wendet D = (1/c * d/dt, i*d/dx, j*d/dy, k*d/dz) auf Psi = Phi/c + A an.

    Phi: Skalarpotential (Ausdruck in t,x,y,z)
    Ax, Ay, Az: Komponenten des Vektorpotentials
    """
    Psi = (Phi / c) + Ax * i + Ay * j + Az * k

    D_t = (1 / c) * diff(Psi, t)
    D_x = i * diff(Psi, x)
    D_y = j * diff(Psi, y)
    D_z = k * diff(Psi, z)

    F = D_t + D_x + D_y + D_z
    res = _simplify_quaternion_element(F)

    print("### Ergebnisse der quaternionischen Feld-Ableitung ###")
    print(f"Gesamt-Feldtensor F: {res}")
    print("-" * 30)

    return res

## Demo

Standard-Szenario aus dem Skript (wie `if __name__ == "__main__":`).

In [ ]:
test_Phi = x**2 + y**2
test_Ax = -y * (c / 2)
test_Ay = x * (c / 2)
test_Az = 0

F_demo = apply_quaternion_operator(test_Phi, test_Ax, test_Ay, test_Az)
F_demo